# Promptable segmentation with SAM

Meta's **Segment Anything Model (SAM)** segments objects it was never trained on, guided by a *prompt*:
a click, a box, or nothing at all ("segment everything"). This notebook walks through the `imgseg`
package built on top of it, using the pictures in `samples/`.

Run `pip install -e .` from the repo root and get the weights first (see the README, *Get the weights*).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from imgseg import Segmenter
from imgseg.io import output_path, save_image
from imgseg.metrics import iou
from imgseg.postprocess import clean_mask, cutout
from imgseg.render import draw_prompts, overlay, overlay_instances

# Works whether Jupyter was started from the repo root or from notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "samples").is_dir())
SAMPLES = ROOT / "samples"
OUT = ROOT / "outputs"  # git-ignored


def show(images, titles=None, cols=None, size=4.6):
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 0.68 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, title in zip(axes.ravel(), images, titles or [""] * len(images)):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


seg = Segmenter()  # CUDA if available, otherwise CPU
print("device:", seg.device)

## 1. Encode once, prompt many times

SAM has two very different parts: a heavy **image encoder** (a ViT) that runs once per image, and a tiny
**prompt encoder + mask decoder** that runs once per prompt. `Segmenter.set_image` runs the encoder and caches
the embedding, so every later `predict` call is cheap. (The first call on a GPU also pays one-off warm-up.)

In [ ]:
seg.set_image(SAMPLES / "football_dribble.jpeg")
first = seg.last_encode_seconds
seg.set_image(SAMPLES / "football_dribble.jpeg")
print(f"encode: {first:.2f}s on the first call, {seg.last_encode_seconds:.2f}s once warm")
show([seg.image], ["samples/football_dribble.jpeg"])

## 2. One click is ambiguous

Click on a shirt: do you mean the shirt, the player, or the player *and* the ball? SAM answers with **three
candidate masks and a predicted-IoU score for each**. For a lone point, `predict` returns all three and
`.mask` is the highest-scoring one. Pass `select="largest"` to take the widest candidate instead. Neither rule is
right for every click: SAM's confidence tends to favour small, confident parts, while "largest" fails when you
want just the shirt. The README's *Evaluation* section measures both on a labelled benchmark.

> The first version of this notebook prompted at 55% width / 50% height of the frame. That is open grass, so SAM
> correctly segmented the *pitch*, and the "isolated object" it exported was the pitch, not a player.

In [ ]:
GRASS, SHIRT = (404, 235), (352, 205)

grass = seg.predict(points=[GRASS], labels=[1])
shirt = seg.predict(points=[SHIRT], labels=[1])

tiles = [draw_prompts(overlay(seg.image, grass.mask), points=[GRASS], labels=[1])]
titles = [f"click on grass -> the pitch ({grass.mask.mean():.0%} of the image)"]
for i, (mask, score) in enumerate(zip(shirt.candidates, shirt.scores)):
    tiles.append(draw_prompts(overlay(seg.image, mask), points=[SHIRT], labels=[1]))
    titles.append(f"shirt click, candidate {i}: score {score:.2f}, {int(mask.sum())} px")
show(tiles, titles, cols=2)

## 3. Several points are *one* prompt

Extra clicks disambiguate. **All points together form a single prompt**: label `1` says "this is part of the
object", label `0` says "this is not". With more than one point `predict` returns a single mask.

(An earlier version passed a flat list of points to Ultralytics, which treats each point as its own separate prompt,
so a background click produced its own foreground mask. Here the points are always combined.)

In [ ]:
BODY = [(352, 205), (330, 250), (395, 262)]  # shirt, shorts, right boot
NEG = (297, 277)                                   # the ball

body = seg.predict(points=BODY, labels=[1, 1, 1])
no_ball = seg.predict(points=BODY + [NEG], labels=[1, 1, 1, 0])
print("masks returned:", body.candidates.shape[0], "and", no_ball.candidates.shape[0])
print(f"with the ball: {int(body.mask.sum())} px | background click on the ball: {int(no_ball.mask.sum())} px")

show(
    [draw_prompts(overlay(seg.image, body.mask), points=BODY, labels=[1, 1, 1]),
     draw_prompts(overlay(seg.image, no_ball.mask), points=BODY + [NEG], labels=[1, 1, 1, 0])],
    ["3 foreground points", "+ 1 background point (red) on the ball"],
)

## 4. Box prompt

A box is usually the most reliable prompt: it states the extent of the object. The embedding from step 1 is
reused, so this costs milliseconds, not another encode.

In [ ]:
import time

BOX = (250, 120, 420, 310)
start = time.perf_counter()
by_box = seg.predict(box=BOX)
ms = (time.perf_counter() - start) * 1000
print(f"box prompt: {ms:.0f} ms (embedding reused) | score {by_box.score:.2f} | "
      f"IoU with the 3-point mask: {iou(by_box.mask, body.mask):.3f}")
show([draw_prompts(overlay(seg.image, by_box.mask), box=BOX)], ["box prompt"])

## 5. Clean up and cut out

`clean_mask` drops fragments smaller than `min_area` pixels (`keep="largest"` keeps only the biggest region;
the default `keep="all"` keeps every region above the threshold so multi-object masks survive).
`cutout` turns a mask into an RGBA image with the mask as its alpha channel, optionally cropped and feathered.

In [ ]:
mask = clean_mask(by_box.mask, min_area=500, keep="largest")
rgba = cutout(seg.image, mask, crop=True, feather=1.5)

# Composite onto grey so the transparency is visible.
alpha = rgba[..., 3:4] / 255.0
preview = (rgba[..., :3] * alpha + 60 * (1 - alpha)).astype(np.uint8)
show([preview], [f"cut-out, cropped to {rgba.shape[1]}x{rgba.shape[0]}"], size=3)

OUT.mkdir(exist_ok=True)
saved = save_image(rgba, output_path(SAMPLES / "football_dribble.jpeg", "box", "cutout", OUT))
print("saved", saved.relative_to(ROOT))

## 6. "Segment everything" finds parts, not objects

With no prompt, SAM places a grid of points and keeps the masks that pass quality filters (predicted IoU and
stability score). `segment_everything` returns them **separately, largest first**. They are never merged into one
mask: a union of dozens of masks is nearly meaningless as "the object".

Notice that the results are *part-level*: SAM does not group a shirt, arms and head into "a person", and it
attaches no labels.

In [ ]:
SCENES = ["trophy_celebration.jpeg", "wolf_of_wall_street_poster.jpg", "football_dribble.jpeg", "starry_night.jpg"]
auto = {}
print(f"{'image':34} {'masks':>5} {'pixels covered':>15} {'largest mask':>13} {'median mask':>12}")
for name in SCENES:
    seg.set_image(SAMPLES / name)
    segments = auto[name] = seg.segment_everything()
    n_pixels = seg.image.shape[0] * seg.image.shape[1]
    covered = np.any([s.mask for s in segments], axis=0).mean()
    areas = np.array([s.area for s in segments]) / n_pixels
    print(f"{name:34} {len(segments):>5} {covered:>15.1%} {areas.max():>13.1%} {np.median(areas):>12.2%}")

tiles, titles = [], []
for name in SCENES[:2]:
    seg.set_image(SAMPLES / name)
    tiles.append(overlay_instances(seg.image, [s.mask for s in auto[name]]))
    titles.append(f"{name}: {len(auto[name])} separate masks")
show(tiles, titles)

## 7. The failure case, measured

The poster (dense illustration, overlapping faces, text) is where segment-everything looks worst. Instead of
guessing why, test it:

* **How much does auto mode cover?** The table above shows how little of the poster the surviving masks cover.
  Cells below relax the quality filters to check whether that is just the filters rejecting ambiguous proposals.
* **Is the poster simply unsegmentable?** No: a box on the central figure gives a sensible mask. But SAM's own
  confidence is lowest there, which makes the score a useful warning signal.

In [ ]:
seg.set_image(SAMPLES / "wolf_of_wall_street_poster.jpg")
n_pixels = seg.image.shape[0] * seg.image.shape[1]
print("segment-everything on the poster as the quality filters are relaxed:")
for iou_t, stab_t in [(0.88, 0.95), (0.80, 0.90), (0.70, 0.80)]:
    segments = seg.segment_everything(pred_iou_thresh=iou_t, stability_score_thresh=stab_t)
    covered = np.any([s.mask for s in segments], axis=0).mean() if segments else 0.0
    print(f"  pred_iou >= {iou_t:.2f}, stability >= {stab_t:.2f}: {len(segments):>3} masks, {covered:.1%} of pixels covered")

In [ ]:
PROMPTED = {
    "wolf_of_wall_street_poster.jpg": ("central figure", dict(box=(215, 85, 345, 235))),
    "starry_night.jpg": ("cypress tree", dict(box=(222, 70, 296, 312))),
    "trophy_celebration.jpeg": ("player lifting the trophy", dict(box=(365, 45, 490, 330))),
}
tiles, titles = [], []
for name, (what, prompt) in PROMPTED.items():
    seg.set_image(SAMPLES / name)
    pred = seg.predict(**prompt)
    tiles.append(draw_prompts(overlay(seg.image, pred.mask), **prompt))
    titles.append(f"{what}: score {pred.score:.2f}")
show(tiles, titles, cols=3)

**What this shows.** Prompted segmentation handles all three images, including the poster. Segment-everything is
the weak mode: at the default thresholds it returns a few dozen part-level fragments that cover a small share of the
poster. Relaxing the two quality thresholds recovers far more of the image (the cell above prints the numbers), so a
large part of the "failure" is the filters rejecting low-confidence proposals. The extra masks are exactly the ones SAM
is least sure about, and without ground-truth masks this notebook cannot say how good they are.

The previous version of this notebook blamed "SAM interpreting contrast changes as object edges" and merged every mask
into one image, which hid all of this. That explanation was never tested; the filter experiment is a partial test of a
different one, not proof of why SAM struggles on dense artwork.

## 8. Making it faster

Measured with `python scripts/benchmark.py --mobile-sam weights/mobile_sam.pt` on **NVIDIA GeForce RTX 4060 Laptop GPU, torch 2.8.0+cu129**: the same image and
box for every row, median of repeated runs after warm-up, CUDA synchronised.

| config | encode ms | prompt ms | encode + prompt ms | peak GPU MB | IoU vs CUDA fp32 |
|---|---:|---:|---:|---:|---:|
| CUDA fp32 | 586 | 38 | 624 | 2770 | 1.000 |
| CUDA fp16 (autocast) | 358 | 53 | 412 | 1761 | 0.999 |
| CUDA MobileSAM (Ultralytics) | n/a | n/a | 109 | 290 | 0.960 |
| CPU fp32 | 13709 | 193 | 13905 | - | 1.000 |
| CPU int8 encoder (dynamic) | 12800 | 194 | 13006 | - | 0.992 |

*IoU vs CUDA fp32* is the overlap of the resulting mask with the fp32 baseline, so each speed-up shows its accuracy cost.
Re-run the script on your own hardware; these numbers are specific to this machine.

Reading the table:

* **Encode once**: the encoder takes 586 ms and a prompt only 38 ms on CUDA fp32, so extra prompts on the same image are nearly free.
* **fp16 autocast** cut encode + prompt from 624 to 412 ms (1.5x) and peak GPU memory from 2770 to 1761 MB, with mask IoU 0.999 against fp32. It only helps the encoder; the prompt step got slightly slower (38 -> 53 ms).
* **Dynamic INT8** on CPU took 13.0 s against 13.9 s for fp32 (6% faster, IoU 0.992): a small gain, nowhere near what moving to a GPU gives.
* **MobileSAM** was the fastest at 109 ms (5.7x faster than CUDA fp32) and used 290 MB, at a mask IoU of 0.960 against the ViT-B mask. That is overlap with another model's mask, not with ground truth. Ultralytics' timer excludes image pre/post-processing, which the imgseg rows include, so the comparison slightly favours MobileSAM.

Corrections to what this notebook said before:

* INT8 here is PyTorch *dynamic* quantisation of the encoder's `Linear` layers, which only runs on CPU and does not apply to
  the CUDA rows. Applying `.half()` first and then INT8 quantisation (as in the old markdown snippet) fails:
  `quantize_dynamic` on half-precision `Linear` layers raises `RuntimeError: expected scalar type Float but found Half`.
* MobileSAM replaces the heavy encoder with a distilled **TinyViT** (the old text said MobileViT). It is measured through Ultralytics,
  which re-encodes on every call, so only the combined time is available.
* Structured pruning was listed before but never implemented, so it is not claimed here.

## How SAM works

SAM is **Image Encoder + Prompt Encoder + Mask Decoder**:

1. **Image encoder**: a Vision Transformer turns the image into a dense embedding. This is the expensive step and runs once per image.
2. **Prompt encoder**: points and boxes become sparse embeddings; an optional low-resolution mask prompt becomes a dense one.
3. **Mask decoder**: a small transformer lets the prompt tokens and the image embedding attend to each other, then outputs masks plus a
   predicted IoU for each. For an ambiguous prompt it returns three masks (whole / part / sub-part) instead of guessing.

It was trained on SA-1B (about 11M images, 1.1B masks), which is why it transfers to new images without retraining.

```
          +-------------------------------+
          |          Input image          |
          +-------------------------------+
                          |
                          v
                 [ Image encoder: ViT ]          <- heavy, once per image
                          |  image embedding
                          |
 points / boxes / mask    |
          |               |
          v               v
   [ Prompt encoder ] -> [ Mask decoder ]        <- light, once per prompt
                          |
                          v
          3 candidate masks + predicted IoU
```

**Limitations.** SAM returns class-agnostic masks (no labels), can fragment dense or textured scenes in
segment-everything mode, and can miss thin structures.

### How it compares with other segmentation models

| Model | Task | Works on unseen images | Needs training data | Notes |
|---|---|---|---|---|
| **U-Net** | Semantic | no | yes | Strong for domain-specific work such as medical imaging |
| **Mask R-CNN** | Instance | only for its trained classes | yes | Detects and labels a fixed set of classes |
| **SAM** | Promptable, class-agnostic | yes | no | Masks without labels; needs a prompt to pick an object |